# Programming exercise 3: Split step Fourier method

Due on Monday, 04.05.26, 20h

<font color = red>

Common issues:
- getting the k-vector right
- expectation value=mean of e.g. x averaged over the probability density $|\psi(x)|^2$
- wave function incorretly normalized. Probability density should be standard Gaussian, not wave function.
- visualization of complex valued functions...
    
    
Possible improvements for next time: 
- Skip the first exercise. Replace it by something examining the output of fft.freq and making sure the correct momentum vector is used for split step.
- Include a more interesting physics case, where the benefit of doing split step becomes more transparent.

<font>

## Defining the problem

We want to calculate the time evolution of a quantum particle in a one-dimensional potential, i.e. solve the initial value problem

$$ i\partial_t \psi(x,t) = \left[-\frac{1}{2} \partial_{x}^2 + V(x,t)\right] \psi(x,t) $$

with initial condition

$$ \psi(x,t=0) = \psi_0(x) $$

by representing the wave function $\psi(x,t)$ on a discrete spatial grid and propagating it using the split-step-Fourier method.

In [ ]:
# load standard libraries

import numpy as np   # standard numerics library

import matplotlib.pyplot as plt   # for making plots

from matplotlib import animation

from IPython.display import HTML

import Comp_Quant_Dynam as cqd # our numerics module

### Exercise 1

Write a function that calculates the discrete Fourier transform of a wave function. Use only even numbers of grid points. You have learned that the split-step-Fourier method automatically imposes periodic boundary conditions, so it makes sense to define the spatial grid like -L/2,-L/2+dx ... L/2-dx.

Hint: Try to avoid for-loops as they slow down your code. Work with the reshape command and use broadcasting where possible. Think about how you want to normalize the Fourier transform.

Test your function by Fourier transforming functions where you know what the outcome should be, e.g. a constant function, a delta function etc.
Implement also the inverse Fourier transform. Make sure you get back the original function after applying the FT and inverse FT.

Numpy provides a fast Fourier transform module. Compare your manual FT the result of the fast Fourier algorithm. Pay special attention the ordering in which the Fourier components are returned by the FFT. You can get the k-values corresponding to the components of the vector returned by fft from np.fft.fftfreq(npoints,d=dx).

### Solution

1) "Manual" Fourier transform.

In [ ]:
# define the grid
L = 20
npoints = 40
xvals, dx = cqd.utility.create_xvals(L, npoints, endpoint=False)

# momenta (assuming even number of points)
kvals = 2 * np.pi / L * np.linspace(-npoints / 2, npoints / 2, npoints, endpoint=False)
dk = 2 * np.pi / L;

# set up test wave function (constant, cos-fct with wave number k)
psi=np.ones(npoints)/np.sqrt(L)
k=1
psi = np.cos(2*np.pi*xvals * k / L) / np.sqrt(L/2) 

# transform
phi=cqd.utility.FT(psi,xvals,kvals)

# transform back
psi2=cqd.utility.iFT(phi,xvals,kvals)

plt.plot(kvals, np.abs(phi) ** 2)
plt.xlabel("k")
plt.ylabel("$|\\phi(k)|^2$")

# compare back and forth transformed to original
print("Max error (position space): ", np.max(np.abs(psi2-psi)))

# print the norm
print("Norm (position space): ", sum(np.abs(psi)**2)*dx)
print("Norm (momentum space): ", sum(np.abs(phi)**2)*dx)


2) Numpy FFT

In [ ]:
# the k-values
kvals2 = 2 * np.pi * np.fft.fftfreq(npoints, d = dx)

# apply FFT and inverse FFT
phi2=np.fft.fft(psi)
psi2=np.fft.ifft(phi2)

# plot and compare to manual FT result
plt.plot(kvals2, np.abs(phi2) ** 2, 'r.')
plt.plot(kvals, np.abs(phi) ** 2)
plt.xlabel("k")
plt.ylabel("$|\\phi(k)|^2$")

# norm in k-space
print("Norm (momentum space): ", sum(np.abs(phi2) ** 2) * dx)

# compare back and forth transformed to original
print("Max error (position space): ", np.max(np.abs(psi-psi2)))

In [ ]:
print("Frequencies np.fft (kvals2): ", kvals2/np.pi)
print("Frequencies manual FT (kvals): ", kvals/np.pi)

### Exercise 2

Implement the split step Fourier algorithm using the numpy FFT and iFFT functions.

Test your code by propagating a Gaussian wave packet in free space. 
Calculate mean and variance at each time and plot them. Does your observation match your expectation?
Below there are some example parameters that you could use.

Animate the time evolution of the wave packet using the method from programming exercise 2.

Optional: Compare the results to the exact analytical solution for a propagating wave packet. Try to increase the spatial and temporal step size to see how the error depends on them. You can also look at the wave packet in Fourier space, where the analytical solution is even simpler. What happens when you make the initial momentum p0 really large? Can you understand this by looking at the wave packet in momentum space?

In [ ]:
# define the grid
#L = 20
#npoints = 256

# parameters of the wave packet
#x0 = -5;
#sigma = 1;
#p0 = 1;

#time steps
#dt = 0.1;
#tsteps = 50;

### Solution

In [ ]:
# define the grid
L = 40
npoints = 1024
xvals, dx = cqd.utility.create_xvals(L, npoints, endpoint=False)
kvals = 2 * np.pi * np.fft.fftfreq(npoints, d = dx)

# prepare a wave packet
#x0 = -10;
#sigma = 2;
#p0 = 1.5;

# coherent state
x0 = -4;
sigma = 1/np.sqrt(2);
p0 = 5;
# note: in this case a time step of 0.1 leads to residual oscillations of the width.

gauss_wp = cqd.utility.gaussian_wave_packet(xvals, x0, sigma, p0)
# check normalization
print("Normalization: ", sum(np.abs(gauss_wp)**2)*dx)

# plot the probability
plt.figure()
plt.plot(xvals, np.abs(gauss_wp) ** 2)
plt.xlabel("x")
plt.ylabel("$|\\psi_0(x)|^2$")

# define the potential
V0 = 1
#V = 0*xvals # free particle
V = cqd.hamiltonians.HO_potential_sparse(xvals).diagonal() # harmonic potential
#V = np.diag(cqd.hamiltonians.step_potential(xvals, V0)) # potential step

# plot the potential
plt.figure()
plt.plot(xvals, V)
plt.xlabel("x")
plt.ylabel("V(x)")    
plt.show()

In [ ]:
# initialize psi
psi0 = gauss_wp

# propagate the wave packet
dt = 0.1
tsteps = 100;
tvec = cqd.utility.create_tvecs(tsteps, dt)

# container for storing the result
V_t = lambda t: V # time-independent potential
psit = cqd.unitaries.t_evol_split_step_fourier(psi0, V_t, tvec, xvals)

In [ ]:
#check norm conservation
print("Normalization at final time: ", (np.linalg.norm(psit, axis=1) ** 2 * dx)[-1])

#plot mean and variance
xmean=np.sum(xvals * np.abs(psit) ** 2 * dx, axis=1)
xvar=np.sum(xvals ** 2 * np.abs(psit) ** 2 * dx, axis=1) - xmean ** 2

plt.figure()
plt.plot(tvec,xmean)
plt.xlabel("t")
plt.ylabel("<x>")
plt.show()

plt.figure()
plt.plot(tvec,xvar)
plt.xlabel("t")
plt.ylabel("$(\\Delta x)^2$")
plt.show()

Make an animation of the wave packet

In [ ]:
%%capture
fig, ax = plt.subplots()
ax.set_xlim(( -L/2, L/2))
ax.set_ylim((0, 1.1))
ax.set_xlabel("x")
ax.set_ylabel("$|\\psi(x,t)|^2$")
line, = ax.plot([],[])


abs_val = lambda t, psit: np.abs(psit[t])**2
args = (abs_val, xvals, line, psit)

anim = animation.FuncAnimation(
    fig,
    cqd.plotting.animate,
    frames=np.arange(tsteps), # t-values,
    fargs=args, # function arguments to be passed to animate
    interval=50, # wait time before displaying new frame in ms
    blit=True
)

In [ ]:
test = (args, (1, 2, 3))
len(test)

In [ ]:
HTML(anim.to_jshtml())

### Exercise 3 (optional)

Experiment with different potentials!

Let the wave packet from before evolve in a harmonic potential (with p0=0). Can you recover the coherent state dynamics from programming exercise 2?
Again, calculate temporal evolution of mean and variance and visualize the wave packet evolution in an animation. Comment on your observations.

Now simulate the scattering off a potential step $V(x)\propto\theta(x)$. Choose different initial velocities (momenta). Interpret your results. What happens when the reflected wave packet reaches the boundary of the spatial grid? Example parameters are given below. Make sure your results are converged with respect to spatial and temporal step size and spatial grid size.

Be creative! Let your wave packet propagate through a potential barrier, across a well, or down a step etc. Measure the reflected and transmitted probability (and the probability to be inside the barrier) as a function of time and study the transmission systematically as a function of the initial momentum. Try out time dependent potentials, such as a barrier of oscillating height or position.

General hint: Pay attention to the data type of vectors. If you use containers for storing the wave function, make sure you initialize them as complex vectors.

In [ ]:
# define the grid
#L = 40
#npoints = 512

# parameters of the wave packet
#x0 = -10;
#sigma = 2;
#p0 = 1;# try also 1.5 and 2

#time steps
#dt = 0.1;
#tsteps = 250;

Time dependent potential

In [ ]:
# define the grid
L = 40
npoints = 512
dx = L/npoints
xvals, dx = cqd.utility.create_xvals(L, npoints, endpoint=False)
kvals = 2 * np.pi * np.fft.fftfreq(npoints, d = dx)

# prepare a wave packet
x0 = -10;
sigma = 2;
p0 = 1.2;

gauss_wp = cqd.utility.gaussian_wave_packet(xvals, x0, sigma, p0)
# check normalization
print("Normalization: ", sum(np.abs(gauss_wp)**2)*dx)

# plot the probability
plt.figure()
plt.plot(xvals,np.abs(gauss_wp)**2)
plt.xlabel("x")
plt.ylabel("$|\\psi_0(x)|^2$")


# define a time dependent potential
well_width = 4
V0 = 1
modulation_amplitude = 0.1
modulation_frequency = 0.01
V_barrier = cqd.hamiltonians.barrier_potential(xvals, V0, well_width) # potential barrier

    
# plot the potential
plt.figure()
plt.plot(xvals, V_barrier)
plt.xlabel("x")
plt.ylabel("V(x)")
plt.show()

In [ ]:

# initialize psi
psi0 = gauss_wp

# propagate the wave packet
dt = 0.1
tsteps = 250;
tvec = cqd.utility.create_tvecs(tsteps, dt)

modulate_V = lambda t: V_barrier * (V0 + modulation_amplitude * np.cos(2 * np.pi * modulation_frequency * t))
psit = cqd.unitaries.t_evol_split_step_fourier(psi0, modulate_V, tvec, xvals)
psit_k = np.fft.fft(psit, axis=1)

In [ ]:
%%capture
fig, ax = plt.subplots()
ax.set_xlim(( -L/2, L/2))
ax.set_ylim((-0., 0.5))
ax.set_xlabel("x")
ax.set_ylabel("$|\psi(x,t)|^2$")
line, = ax.plot([],[])
line2, = ax.plot([],[])

abs_val = lambda t, psit: np.abs(psit[t])**2
args = (abs_val, xvals, line, psit)

V_t = lambda t, pot: 0.3 * pot(t)
args_V = (V_t,  xvals, line2, modulate_V)

args_all = (args, args_V)

anim = animation.FuncAnimation(
    fig,
    cqd.plotting.multi_animate,
    frames=np.arange(tsteps), # t-values,
    fargs=args_all, # function arguments to be passed to animate
    interval=50, # wait time before displaying new frame in ms
    blit=True
)

In [ ]:
HTML(anim.to_jshtml())

Plot in momentum space

In [ ]:
%%capture
fig, ax = plt.subplots()
ax.set_xlim((-L/2, L/2))
ax.set_ylim((0, .1))
ax.set_xlabel("k")
ax.set_ylabel("$|\\phi(k,t)|^2$")
line, = ax.plot([],[])


abs_val = lambda t, psit, dx, npoints: np.abs(psit[t]) ** 2 * dx / npoints
args = (abs_val, kvals, line, psit_k, dx, npoints)

anim = animation.FuncAnimation(
    fig,
    cqd.plotting.animate,
    frames=np.arange(tsteps), # t-values,
    fargs=args, # function arguments to be passed to animate
    interval=50, # wait time before displaying new frame in ms
    blit=True
)

In [ ]:
HTML(anim.to_jshtml())

Final frame:

In [ ]:
it=-1
plt.plot(kvals,np.abs(psit_k[it])**2*dx/xvals.size)
plt.ylim((0, .04))
plt.xlim((-10,10))
plt.xlabel("k")
plt.ylabel("$|\\phi(k,t)|^2$")
plt.show()